## Copy random NDVI data to home

In [4]:
%%bash
set -euo pipefail

# ---- EDIT THIS ----
TILE_ID="p089r078"
BUCKET="dcceew-eds-data"
PREFIX="AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/optimised/tiles/${TILE_ID}/ndvi/"
DEST_ROOT="/home/jovyan/eds-outputs"
N=5
# -------------------

DEST_DIR="${DEST_ROOT}/${TILE_ID}_ndvi"
mkdir -p "$DEST_DIR"

echo "[INFO] Destination: $DEST_DIR"
echo "[INFO] Listing NDVI objects under s3://${BUCKET}/${PREFIX}"

# 1) List all NDVI tifs (keys) under the prefix
mapfile -t NDVI_KEYS < <(
  aws s3api list-objects-v2 \
    --bucket "$BUCKET" \
    --prefix "$PREFIX" \
    --query "Contents[?ends_with(Key, '_ndvi_28356.tif') || contains(Key, '_ndvi_')].Key" \
    --output text \
  | tr '\t' '\n' \
  | grep -E '_ndvi_.*\.tif$' \
  | sort -u
)

echo "[INFO] Found ${#NDVI_KEYS[@]} NDVI .tif objects"

if [ "${#NDVI_KEYS[@]}" -eq 0 ]; then
  echo "[ERROR] No NDVI keys found. Check TILE_ID/PREFIX or naming."
  echo "        Example prefix should look like: .../tiles/${TILE_ID}/ndvi/"
  exit 1
fi

# 2) Pick N random NDVI keys
mapfile -t PICKED < <(printf "%s\n" "${NDVI_KEYS[@]}" | shuf -n "$N")

echo "[INFO] Downloading $N random NDVI+FFMASK pairs..."

# 3) Download NDVI + corresponding FFMASK
for key in "${PICKED[@]}"; do
  ndvi_name="$(basename "$key")"
  ffmask_key="${key/_ndvi_/_ffmask_}"
  ffmask_name="$(basename "$ffmask_key")"

  echo "  - NDVI:   $ndvi_name"
  aws s3 cp "s3://${BUCKET}/${key}" "${DEST_DIR}/${ndvi_name}"

  echo "  - FFMASK: $ffmask_name"
  aws s3 cp "s3://${BUCKET}/${ffmask_key}" "${DEST_DIR}/${ffmask_name}"
done

echo
echo "[OK] Done. Files now in: $DEST_DIR"
ls -lh "$DEST_DIR" | head -n 50

[INFO] Destination: /home/jovyan/eds-outputs/p089r078_ndvi
[INFO] Listing NDVI objects under s3://dcceew-eds-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/optimised/tiles/p089r078/ndvi/
[INFO] Found 232 NDVI .tif objects
[INFO] Downloading 5 random NDVI+FFMASK pairs...
  - NDVI:   lztmre_p089r078_20230423_ndvi_28356.tif
download: s3://dcceew-eds-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/optimised/tiles/p089r078/ndvi/L8/2023/20230423/lztmre_p089r078_20230423_ndvi_28356.tif to ../../../eds-outputs/p089r078_ndvi/lztmre_p089r078_20230423_ndvi_28356.tif
  - FFMASK: lztmre_p089r078_20230423_ffmask_28356.tif
download: s3://dcceew-eds-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/optimised/tiles/p089r078/ndvi/L8/2023/20230423/lztmre_p089r078_20230423_ffmask_28356.tif to ../../../eds-outputs/p089r078_ndvi/lztmre_p089r078_20230423_ffmask_28356.tif
  - NDVI:   lztmre_p089r078_20161129_ndvi_28356.tif
download: s3://dcceew-eds-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/optimised/tiles/p089r078/n